# Stage 4 ? Quran phonetic annotation and label validation

Read-only exploration of a bounded sample from multiple configurations. This notebook does not tokenize the full corpus, change labels, train, or create splits. Run from the repository root after installing the project and notebook extras (`python -m pip install -e ".[notebook]"`). Network access to the public Hugging Face dataset-server API is required. Rows and audio payloads are not downloaded as a corpus.

## 1. Dataset information

Dataset and upstream implementation facts are summarized from the local Stage 1?3 reports and source paths cited in `docs/annotation_validation_report.md`.

In [1]:
import json
import requests
import pandas as pd
import time
from quran_transcript import chunck_phonemes
from quran_annotations import validate_record

DATASET = "obadx/muaalem-annotated-v3"
API = "https://datasets-server.huggingface.co/rows"
def fetch(config, offset=0, length=5):
    params = {"dataset": DATASET, "config": config,
        "split": "train", "offset": offset, "length": length}
    for attempt in range(4):
        try:
            response = requests.get(API, params=params, timeout=90)
            response.raise_for_status()
            return [r["row"] for r in response.json()["rows"]]
        except requests.RequestException:
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)

configs = ["moshaf_0.1", "moshaf_1.0", "moshaf_26.1"]


## 2. Available subsets and splits

Stage 1 datasets-server metadata reported 27 recitation configurations plus `moshaf_metadata` and `reciters_metadata`, each with a `train` split only. The following bounded sample requests test three recitation configurations now; counts and full configuration list are recorded in the Stage 1 report.

In [2]:
sample_by_config = {config: fetch(config, length=5) for config in configs}
{config: len(rows) for config, rows in sample_by_config.items()}

{'moshaf_0.1': 5, 'moshaf_1.0': 5, 'moshaf_26.1': 5}

## 3. Dataset schema

In [3]:
schema_rows = []
for config, rows in sample_by_config.items():
    schema_rows.extend({"configuration": config, "field": k,
                        "observed_python_type": type(v).__name__,
                        "is_null": v is None}
                       for k, v in rows[0].items())
schema = pd.DataFrame(schema_rows)
schema

,configuration,field,observed_python_type,is_null
0,moshaf_0.1,audio,list,False
1,moshaf_0.1,segment_index,str,False
2,moshaf_0.1,tarteel_transcript,list,False
3,moshaf_0.1,moshaf_id,str,False
4,moshaf_0.1,moshaf_name,str,False
...,...,...,...,...
67,moshaf_26.1,start_span,NoneType,True
68,moshaf_26.1,end_span,NoneType,True
69,moshaf_26.1,match_ratio,float,False
70,moshaf_26.1,phonemes,str,False


## 4. Sample inspection

In [4]:
records = [(config, row) for config, rows in sample_by_config.items() for row in rows]
examples = []
for config, row in records:
    result = validate_record(row)
    examples.append({"configuration": config,
        "segment_index": row.get("segment_index"), "uthmani": row.get("uthmani"),
        "imlaey": row.get("imlaey"), "phonemes": row.get("phonemes"),
        "sifat_count": len(row.get("sifat") or []),
        "sifat_first_entry": (row.get("sifat") or [None])[0],
        "start_span": row.get("start_span"), "end_span": row.get("end_span"),
        "has_quran": row.get("has_quran"), "has_istiaatha": row.get("has_istiaatha"),
        "has_bismillah": row.get("has_bismillah"), "has_sadaka": row.get("has_sadaka"),
        "validation": result.to_dict()})
pd.DataFrame(examples).head(6)

,configuration,segment_index,uthmani,imlaey,phonemes,sifat_count,sifat_first_entry,start_span,end_span,has_quran,has_istiaatha,has_bismillah,has_sadaka,validation
0,moshaf_0.1,001.0000,أَعُوذُ بِٱللَّهِ مِنَ ٱلشَّيْطَانِ ٱلرَّجِيمِ,أَعُوذُ بِاللَّهِ مِنَ الشَّيْطَانِ الرَّجِيمِ,ءَعُۥۥذُ بِللَااهِ مِنَ ششَيطَاانِ ررَجِۦۦۦۦم,19,"{'ghonna': 'not_maghnoon', 'hams_or_jahr': 'ja...",None,None,False,True,False,False,"{'is_valid': True, 'errors': [], 'warnings': []}"
1,moshaf_0.1,001.0001,بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ,بِسمِ للَااهِ ررَحمَاانِ ررَحِۦۦۦۦم,15,"{'ghonna': 'not_maghnoon', 'hams_or_jahr': 'ja...","{'sura_idx': 1, 'aya_idx': 1, 'imlaey': 0, 'ut...","{'sura_idx': 1, 'aya_idx': 1, 'imlaey': 4, 'ut...",True,False,False,False,"{'is_valid': True, 'errors': [], 'warnings': []}"
2,moshaf_0.1,001.0002,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَـٰلَمِينَ,الْحَمْدُ لِلَّهِ رَبِّ الْعَالَمِينَ,ءَلحَمدُ لِللَااهِ رَببِ لعَاالَمِۦۦۦۦن,18,"{'ghonna': 'not_maghnoon', 'hams_or_jahr': 'ja...","{'sura_idx': 1, 'aya_idx': 2, 'imlaey': 0, 'ut...","{'sura_idx': 1, 'aya_idx': 2, 'imlaey': 4, 'ut...",True,False,False,False,"{'is_valid': True, 'errors': [], 'warnings': []}"
3,moshaf_0.1,001.0003,ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ,الرَّحْمَٰنِ الرَّحِيمِ,ءَررَحمَاانِ ررَحِۦۦۦۦم,10,"{'ghonna': 'not_maghnoon', 'hams_or_jahr': 'ja...","{'sura_idx': 1, 'aya_idx': 3, 'imlaey': 0, 'ut...","{'sura_idx': 1, 'aya_idx': 3, 'imlaey': 2, 'ut...",True,False,False,False,"{'is_valid': True, 'errors': [], 'warnings': []}"
4,moshaf_0.1,001.0004,مَـٰلِكِ يَوْمِ ٱلدِّينِ,مَالِكِ يَوْمِ الدِّينِ,مَاالِكِ يَومِ ددِۦۦۦۦن,10,"{'ghonna': 'maghnoon', 'hams_or_jahr': 'jahr',...","{'sura_idx': 1, 'aya_idx': 4, 'imlaey': 0, 'ut...","{'sura_idx': 1, 'aya_idx': 4, 'imlaey': 3, 'ut...",True,False,False,False,"{'is_valid': True, 'errors': [], 'warnings': []}"
5,moshaf_1.0,001.0000,أَعُوذُ بِٱللَّهِ مِنَ ٱلشَّيْطَانِ ٱلرَّجِيمِ,أَعُوذُ بِاللَّهِ مِنَ الشَّيْطَانِ الرَّجِيمِ,ءَعُۥۥذُ بِللَااهِ مِنَ ششَيطَاانِ ررَجِۦۦم,19,"{'ghonna': 'not_maghnoon', 'hams_or_jahr': 'ja...",None,None,False,True,False,False,"{'is_valid': True, 'errors': [], 'warnings': []}"


## 5. Audio inspection

This stage is annotation-focused. Audio decoding is covered by the Stage 3 loader and preprocessing notebook. The rows API returns audio as a URL descriptor rather than decoded samples; to avoid extra asset requests, this notebook reports the dataset fields only. Stage 1/3 decoded limited audio examples and documented the feature declaration. No audio transformations occur here.

In [5]:
[{"configuration": c, "segment_index": r.get("segment_index"),
  "audio_present": r.get("audio") is not None, "audio_type": (r.get("audio") or [{}])[0].get("type") if isinstance(r.get("audio"), list) else type(r.get("audio")).__name__,
  "sample_rate_metadata": r.get("sample_rate"),
  "duration_seconds": r.get("duration_seconds")}
 for c, r in records[:3]]

[{'configuration': 'moshaf_0.1',
  'segment_index': '001.0000',
  'audio_present': True,
  'audio_type': 'audio/wav',
  'sample_rate_metadata': 16000,
  'duration_seconds': 6.059999942779541},
 {'configuration': 'moshaf_0.1',
  'segment_index': '001.0001',
  'audio_present': True,
  'audio_type': 'audio/wav',
  'sample_rate_metadata': 16000,
  'duration_seconds': 6.019999980926514},
 {'configuration': 'moshaf_0.1',
  'segment_index': '001.0002',
  'audio_present': True,
  'audio_type': 'audio/wav',
  'sample_rate_metadata': 16000,
  'duration_seconds': 6.28000020980835}]

## 6. Annotation inspection

In [6]:
[(e["configuration"], e["segment_index"], e["uthmani"], e["phonemes"],
  e["sifat_count"], e["sifat_first_entry"], e["validation"]) for e in examples[:4]]

[('moshaf_0.1',
  '001.0000',
  'أَعُوذُ بِٱللَّهِ مِنَ ٱلشَّيْطَانِ ٱلرَّجِيمِ',
  'ءَعُۥۥذُ بِللَااهِ مِنَ ششَيطَاانِ ررَجِۦۦۦۦم',
  19,
  {'ghonna': 'not_maghnoon',
   'hams_or_jahr': 'jahr',
   'istitala': 'not_mostateel',
   'itbaq': 'monfateh',
   'phonemes': 'ءَ',
   'qalqla': 'not_moqalqal',
   'safeer': 'no_safeer',
   'shidda_or_rakhawa': 'shadeed',
   'tafashie': 'not_motafashie',
   'tafkheem_or_taqeeq': 'moraqaq',
   'tikraar': 'not_mokarar'},
  {'is_valid': True, 'errors': [], 'warnings': []}),
 ('moshaf_0.1',
  '001.0001',
  'بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ',
  'بِسمِ للَااهِ ررَحمَاانِ ررَحِۦۦۦۦم',
  15,
  {'ghonna': 'not_maghnoon',
   'hams_or_jahr': 'jahr',
   'istitala': 'not_mostateel',
   'itbaq': 'monfateh',
   'phonemes': 'بِ',
   'qalqla': 'not_moqalqal',
   'safeer': 'no_safeer',
   'shidda_or_rakhawa': 'shadeed',
   'tafashie': 'not_motafashie',
   'tafkheem_or_taqeeq': 'moraqaq',
   'tikraar': 'not_mokarar'},
  {'is_valid': True, 'errors': [], 'warnin

## 7. Reciter and recording metadata

In [7]:
[{"configuration": c, "moshaf_id": r.get("moshaf_id"),
  "moshaf_name": r.get("moshaf_name"), "reciter_id": r.get("reciter_id"),
  "reciter_arabic_name": r.get("reciter_arabic_name"),
  "reciter_english_name": r.get("reciter_english_name"),
  "index_type": r.get("index_type"), "sura_or_aya_index": r.get("sura_or_aya_index"),
  "timestamp_seconds": r.get("timestamp_seconds"),
  "duration_seconds": r.get("duration_seconds"), "match_ratio": r.get("match_ratio")}
 for c, r in records[:5]]

[{'configuration': 'moshaf_0.1',
  'moshaf_id': '0.1',
  'moshaf_name': 'المصحف المرتل بمد المنفصل',
  'reciter_id': 0,
  'reciter_arabic_name': 'محمود خليل الحصري',
  'reciter_english_name': 'Mahmoud Khalil Al-Husari',
  'index_type': 'sura',
  'sura_or_aya_index': '001',
  'timestamp_seconds': [0.0, 6.059999942779541],
  'duration_seconds': 6.059999942779541,
  'match_ratio': 0.95652174949646},
 {'configuration': 'moshaf_0.1',
  'moshaf_id': '0.1',
  'moshaf_name': 'المصحف المرتل بمد المنفصل',
  'reciter_id': 0,
  'reciter_arabic_name': 'محمود خليل الحصري',
  'reciter_english_name': 'Mahmoud Khalil Al-Husari',
  'index_type': 'sura',
  'sura_or_aya_index': '001',
  'timestamp_seconds': [5.739999771118164, 11.760000228881836],
  'duration_seconds': 6.019999980926514,
  'match_ratio': 1.0},
 {'configuration': 'moshaf_0.1',
  'moshaf_id': '0.1',
  'moshaf_name': 'المصحف المرتل بمد المنفصل',
  'reciter_id': 0,
  'reciter_arabic_name': 'محمود خليل الحصري',
  'reciter_english_name': 'Mahmo

## 8. Data quality checks

In [8]:
quality = []
seen = set()
for config, row in records:
    key = (config, row.get("segment_index"))
    duplicate_in_sample = key in seen
    seen.add(key)
    result = validate_record(row)
    quality.append({"configuration": config, "segment_index": row.get("segment_index"),
        "duplicate_identifier_in_sample": duplicate_in_sample,
        "phonemes_chars": len(row.get("phonemes") or ""),
        "sifat_count": len(row.get("sifat") or []),
        "phoneme_groups_equal_sifat": chunck_phonemes(row.get("phonemes") or "") ==
            [item.get("phonemes") for item in (row.get("sifat") or [])],
        "errors": [i.code for i in result.errors],
        "warnings": [i.code for i in result.warnings]})
pd.DataFrame(quality)

,configuration,segment_index,duplicate_identifier_in_sample,phonemes_chars,sifat_count,phoneme_groups_equal_sifat,errors,warnings
0,moshaf_0.1,001.0000,False,45,19,True,[],[]
1,moshaf_0.1,001.0001,False,35,15,True,[],[]
2,moshaf_0.1,001.0002,False,39,18,True,[],[]
3,moshaf_0.1,001.0003,False,23,10,True,[],[]
4,moshaf_0.1,001.0004,False,23,10,True,[],[]
5,moshaf_1.0,001.0000,False,43,19,True,[],[]
6,moshaf_1.0,001.0001,False,33,15,True,[],[]
7,moshaf_1.0,001.0002,False,37,18,True,[],[]
8,moshaf_1.0,001.0003,False,21,10,True,[],[]
9,moshaf_1.0,001.0004,False,21,10,True,[],[]


### Include a flagged non-Quran bismillah example when it is near the beginning

A bounded scan requests at most 200 metadata rows of `moshaf_0.1` to find the first `has_bismillah` record. Audio remains a URL descriptor; no audio asset is downloaded.

In [9]:
bismillah = None
for offset in range(0, 200, 50):
    page = fetch("moshaf_0.1", offset=offset, length=50)
    bismillah = next((r for r in page if r.get("has_bismillah") is True), None)
    if bismillah is not None:
        break
if bismillah:
    result = validate_record(bismillah)
    bismillah_summary = {k: bismillah.get(k) for k in ["segment_index", "imlaey", "uthmani", "phonemes",
       "has_quran", "has_istiaatha", "has_bismillah", "has_sadaka",
       "start_span", "end_span", "match_ratio", "timestamp_seconds", "duration_seconds"]}
    bismillah_summary.update({"sifat_count": len(bismillah.get("sifat") or []),
       "validation": result.to_dict()})
else:
    bismillah_summary = "No flagged bismillah row found in the first 200 rows; see report limitations."
bismillah_summary

{'segment_index': '002.0000',
 'imlaey': 'بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ',
 'uthmani': 'بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ',
 'phonemes': 'بِسمِ للَااهِ ررَحمَاانِ ررَحِۦۦۦۦم',
 'has_quran': False,
 'has_istiaatha': False,
 'has_bismillah': True,
 'has_sadaka': False,
 'start_span': None,
 'end_span': None,
 'match_ratio': 1.0,
 'timestamp_seconds': [0.0, 6.159999847412109],
 'duration_seconds': 6.159999847412109,
 'sifat_count': 15,
 'validation': {'is_valid': True, 'errors': [], 'warnings': []}}

## 9. Findings and conclusions

The report documents validated versus unresolved semantics. In particular, `start_span`/`end_span` coordinate values are structurally checked only: exact offset units and inclusive/exclusive convention were not established from the available original project source. This notebook validates a small sample only and makes no corpus-wide quality claim.